# Bronze Layer - Kafka to HDFS Iceberg (via DBT)

## 🌐 Service UIs (Click These!)
- **HDFS UI**: http://localhost:9870 | **File Browser**: http://localhost:9870/explorer.html#/lakehouse
- **Trino (Query)**: http://localhost:8083
- **Airflow (Orchestration)**: http://localhost:8090 (user: airflow, password: airflow)
- **Spark Master** (Docker): http://localhost:8082 | **Spark UI** (local session): http://localhost:4040
- **Grafana (Monitoring)**: http://localhost:3000 (user: admin, password: admin)
- **Prometheus (Metrics)**: http://localhost:9090
- **Schema Registry**: http://localhost:8081
- **PostgreSQL**: localhost:5432 (user: airflow, password: airflow, db: gold_layer)
- **Kafka Brokers**: localhost:19092, localhost:19093, localhost:19094

**📖 For detailed service info, see** [START_HERE.md](../START_HERE.md)

**💡 To avoid HDFS "RPC response has invalid length"**, run this notebook inside Docker: `docker compose -f docker/docker-compose.yml up -d notebook` then open http://localhost:8888

This notebook ingests data from Kafka topics into HDFS Iceberg tables with Snappy Parquet compression:
- `bookings_raw` → `hdfs://namenode:9000/lakehouse/bronze/bookings_raw`
- `booking_events_raw` → `hdfs://namenode:9000/lakehouse/bronze/booking_events_raw`
- `hotels_raw` → `hdfs://namenode:9000/lakehouse/reference/hotels`

In [9]:
import sys
import os
import json
import time
import os; os.environ.setdefault("LAKEHOUSE_WAREHOUSE", "hdfs://namenode:9000/lakehouse")
sys.path.append('../')

# Set Java 17 for PySpark when on host (Docker image already has JAVA_HOME)
if os.path.exists("/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"):
    os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home'

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *



## 1. Pre-Flight Checks

**🔍 Checking HDFS and Hadoop versions before connecting...**

In [10]:
# Warehouse: use LAKEHOUSE_WAREHOUSE in Docker (HDFS). Default hdfs://namenode:9000/lakehouse for cluster dev; override with env var for local dev.
WAREHOUSE = os.environ.get("LAKEHOUSE_WAREHOUSE", "hdfs://namenode:9000/lakehouse")

# FIX: Force full Hadoop 3.3.6 client stack to match Docker HDFS (avoids "RPC response has invalid length")
# If RPC error persists: restart kernel, then run this cell first. Optionally clear Ivy cache: rm -rf ~/.ivy2/cache/org.apache.hadoop
HADOOP_VERSION = "3.3.6"

print("🔍 Checking versions and connectivity...\n")

# Check Docker HDFS version
import subprocess
try:
    docker_hadoop = subprocess.run(
        ['docker', 'exec', 'namenode', 'hadoop', 'version'],
        capture_output=True, text=True, timeout=5
    )
    docker_hadoop_version = docker_hadoop.stdout.split('\n')[0] if docker_hadoop.returncode == 0 else "Failed"
    print(f"📦 Docker HDFS Hadoop: {docker_hadoop_version}")
except Exception as e:
    print(f"❌ Docker HDFS check failed: {e}")
    
# Check local Hadoop (if available)
try:
    local_hadoop = subprocess.run(['hadoop', 'version'], capture_output=True, text=True, timeout=5)
    local_hadoop_version = local_hadoop.stdout.split('\n')[0] if local_hadoop.returncode == 0 else "Not installed"
    print(f"📦 Local Hadoop: {local_hadoop_version}")
except:
    print(f"📦 Local Hadoop: Not installed (using PySpark's embedded Hadoop)")
    
print()

spark = SparkSession.builder \
    .appName("Bronze-Layer-HDFS") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-client-api:3.3.6,org.apache.hadoop:hadoop-client-runtime:3.3.6,org.apache.hadoop:hadoop-client:3.3.6,org.apache.hadoop:hadoop-hdfs-client:3.3.6,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", WAREHOUSE) \
    .config("spark.sql.iceberg.compression-codec", "snappy") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.iceberg.write.format.default", "parquet") \
    .getOrCreate()

print("✅ Spark session initialized!")
print(f"   Version: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   Warehouse: {WAREHOUSE}")
print(f"   Compression: Snappy Parquet")



🔍 Checking versions and connectivity...

❌ Docker HDFS check failed: [Errno 2] No such file or directory: 'docker'
📦 Local Hadoop: Not installed (using PySpark's embedded Hadoop)

✅ Spark session initialized!
   Version: 3.5.0
   Master: local[*]
   Warehouse: hdfs://namenode:9000/lakehouse
   Compression: Snappy Parquet


## 2. Create Bronze and Reference Databases

In [11]:

# Create databases (Iceberg namespaces on HDFS)
# If running from host, HDFS RPC version mismatch can cause "RPC response has invalid length".
# Fallback: create directories via Docker HDFS CLI so dbt/jobs in Docker can use them.

import subprocess

def create_db_via_hdfs_cli():
    
    """Create lakehouse namespaces via Docker HDFS CLI when Spark RPC fails."""
    for path in ["/lakehouse/bronze", "/lakehouse/reference"]:
        r = subprocess.run(
            ["docker", "exec", "namenode", "hdfs", "dfs", "-mkdir", "-p", path],
            capture_output=True, text=True, timeout=10
        )
        if r.returncode == 0:
            print(f"  Created {path} via HDFS CLI")
        else:
            print(f"  HDFS CLI mkdir {path}: {r.stderr or r.stdout}")

rpc_err = False
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS local.bronze")
    print("✅ Bronze database created")
except Exception as e:
    err_str = str(e)
    print(f"❌ Bronze database failed: {err_str[:200]}...")
    if "RpcException" in err_str or "invalid length" in err_str:
        rpc_err = True
        print("   Fallback: creating /lakehouse/bronze and /lakehouse/reference via HDFS CLI...")
        create_db_via_hdfs_cli()
        print("   Run the rest of the pipeline from Docker (e.g. dbt run) to avoid RPC mismatch.")

if not rpc_err:
    try:
        spark.sql("CREATE DATABASE IF NOT EXISTS local.reference")
        print("✅ Reference database created")
    except Exception as e:
        print(f"❌ Reference database failed: {e}")
        if "RpcException" in str(e) or "invalid length" in str(e):
            create_db_via_hdfs_cli()

print("\n✅ Databases / namespaces ready:")

try:
    spark.sql("SHOW DATABASES").show()
except Exception as e:
    print(f"❌ SHOW DATABASES failed (expected if RPC mismatch): {e}")

✅ Bronze database created
✅ Reference database created

✅ Databases / namespaces ready:
+---------+
|namespace|
+---------+
|  default|
+---------+



## 3. Ingest Bookings from Kafka to HDFS Iceberg

In [12]:
# Define schema for bookings
bookings_schema = StructType([
    StructField("booking_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("hotel_id", StringType(), False),
    StructField("status", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("created_at", StringType(), False),
    StructField("updated_at", StringType(), False)
])

# Read from Kafka
bookings_kafka_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")) \
    .option("subscribe", "bookings_raw") \
    .option("startingOffsets", "earliest") \
    .option("kafka.request.timeout.ms", "120000") \
    .load()

# Parse JSON from Kafka value
bookings_df = bookings_kafka_df.select(
    from_json(col("value").cast("string"), bookings_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    col("data.*"),
    to_timestamp(col("data.created_at")).alias("created_at_ts"),
    to_timestamp(col("data.updated_at")).alias("updated_at_ts"),
    col("kafka_timestamp").alias("processing_ts")
)

print(f"Read {bookings_df.count()} bookings from Kafka")
bookings_df.show(5)

# Write to HDFS Iceberg with Snappy Parquet
bookings_df.writeTo("local.bronze.bookings_raw") \
    .using("iceberg") \
    .tableProperty("write.format.default", "parquet") \
    .tableProperty("write.parquet.compression-codec", "snappy") \
    .partitionedBy("created_at_ts") \
    .createOrReplace()

print("\n✅ Bookings written to HDFS Iceberg:")
print("   📁 Path: hdfs://localhost:9000/lakehouse/bronze/bookings_raw")
print("   📦 Format: Iceberg (Snappy Parquet)")
print(f"   📊 Records: {bookings_df.count()}")

26/02/21 21:21:34 WARN KafkaAdminClient: [AdminClient clientId=adminclient-13] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:34 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Read 150 bookings from Kafka


26/02/21 21:21:35 WARN KafkaAdminClient: [AdminClient clientId=adminclient-14] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:35 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/02/21 21:21:36 WARN KafkaAdminClient: [AdminClient clientId=adminclient-15] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:36 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+----------+-------+--------+---------+-----+-------------------+-------------------+-------------------+-------------------+--------------------+
|booking_id|user_id|hotel_id|   status|price|         created_at|         updated_at|      created_at_ts|      updated_at_ts|       processing_ts|
+----------+-------+--------+---------+-----+-------------------+-------------------+-------------------+-------------------+--------------------+
|        b1|     u1|      h1|  created|120.0|2025-01-01T10:05:00|2025-01-01T10:05:00|2025-01-01 10:05:00|2025-01-01 10:05:00|2026-02-16 02:44:...|
|        b1|     u1|      h1|confirmed|120.0|2025-01-01T10:05:00|2025-01-01T10:10:00|2025-01-01 10:05:00|2025-01-01 10:10:00|2026-02-16 02:44:...|
|        b2|     u2|      h2|  created|200.0|2025-01-01T11:00:00|2025-01-01T11:00:00|2025-01-01 11:00:00|2025-01-01 11:00:00|2026-02-16 02:44:...|
|        b2|     u2|      h2|cancelled|200.0|2025-01-02T09:00:00|2025-01-02T09:05:00|2025-01-02 09:00:00|2025-01-02 09


✅ Bookings written to HDFS Iceberg:
   📁 Path: hdfs://localhost:9000/lakehouse/bronze/bookings_raw
   📦 Format: Iceberg (Snappy Parquet)


26/02/21 21:21:40 WARN KafkaAdminClient: [AdminClient clientId=adminclient-16] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


   📊 Records: 150


## 4. Ingest Booking Events from Kafka to HDFS Iceberg

In [13]:
# Define schema for booking events
events_schema = StructType([
    StructField("booking_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("event_ts", StringType(), False)
])

# Read from Kafka
events_kafka_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")) \
    .option("subscribe", "booking_events_raw") \
    .option("startingOffsets", "earliest") \
    .option("kafka.request.timeout.ms", "120000") \
    .load()

# Parse JSON
events_df = events_kafka_df.select(
    from_json(col("value").cast("string"), events_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    col("data.*"),
    to_timestamp(col("data.event_ts")).alias("event_ts_parsed"),
    col("kafka_timestamp").alias("processing_ts")
)

print(f"Read {events_df.count()} events from Kafka")
events_df.show(5)

# Write to HDFS Iceberg
events_df.writeTo("local.bronze.booking_events_raw") \
    .using("iceberg") \
    .tableProperty("write.format.default", "parquet") \
    .tableProperty("write.parquet.compression-codec", "snappy") \
    .partitionedBy("event_ts_parsed") \
    .createOrReplace()

print("\n✅ Events written to HDFS Iceberg:")
print("   📁 Path: hdfs://localhost:9000/lakehouse/bronze/booking_events_raw")
print("   📦 Format: Iceberg (Snappy Parquet)")
print(f"   📊 Records: {events_df.count()}")

26/02/21 21:21:47 WARN KafkaAdminClient: [AdminClient clientId=adminclient-17] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:47 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/02/21 21:21:47 WARN KafkaAdminClient: [AdminClient clientId=adminclient-18] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:47 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Read 165 events from Kafka


26/02/21 21:21:48 WARN KafkaAdminClient: [AdminClient clientId=adminclient-19] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:48 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+----------+----------+-------------------+-------------------+--------------------+
|booking_id|event_type|           event_ts|    event_ts_parsed|       processing_ts|
+----------+----------+-------------------+-------------------+--------------------+
|        b1|   created|2025-01-01T10:05:00|2025-01-01 10:05:00|2026-02-16 02:44:...|
|        b1| confirmed|2025-01-01T10:10:00|2025-01-01 10:10:00|2026-02-16 02:44:...|
|        b2|   created|2025-01-01T11:00:00|2025-01-01 11:00:00|2026-02-16 02:44:...|
|        b2| cancelled|2025-01-02T09:05:00|2025-01-02 09:05:00|2026-02-16 02:44:...|
|        b4|   created|2025-01-03T18:00:00|2025-01-03 18:00:00|2026-02-16 02:44:...|
+----------+----------+-------------------+-------------------+--------------------+
only showing top 5 rows




✅ Events written to HDFS Iceberg:
   📁 Path: hdfs://localhost:9000/lakehouse/bronze/booking_events_raw
   📦 Format: Iceberg (Snappy Parquet)


26/02/21 21:21:53 WARN KafkaAdminClient: [AdminClient clientId=adminclient-20] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


   📊 Records: 165


## 5. Ingest Hotels from Kafka to HDFS Iceberg

In [14]:
# Define schema for hotels
hotels_schema = StructType([
    StructField("hotel_id", StringType(), False),
    StructField("city", StringType(), False),
    StructField("star_rating", IntegerType(), False)
])

# Read from Kafka
hotels_kafka_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")) \
    .option("subscribe", "hotels_raw") \
    .option("startingOffsets", "earliest") \
    .option("kafka.request.timeout.ms", "120000") \
    .load()

# Parse JSON
hotels_df = hotels_kafka_df.select(
    from_json(col("value").cast("string"), hotels_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    col("data.*"),
    col("kafka_timestamp").alias("processing_ts")
)

print(f"Read {hotels_df.count()} hotels from Kafka")
hotels_df.show()

# Write to HDFS Iceberg
hotels_df.writeTo("local.reference.hotels") \
    .using("iceberg") \
    .tableProperty("write.format.default", "parquet") \
    .tableProperty("write.parquet.compression-codec", "snappy") \
    .createOrReplace()

print("\n✅ Hotels written to HDFS Iceberg:")
print("   📁 Path: hdfs://localhost:9000/lakehouse/reference/hotels")
print("   📦 Format: Iceberg (Snappy Parquet)")
print(f"   📊 Records: {hotels_df.count()}")

26/02/21 21:21:54 WARN KafkaAdminClient: [AdminClient clientId=adminclient-21] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:54 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Read 45 hotels from Kafka


26/02/21 21:21:54 WARN KafkaAdminClient: [AdminClient clientId=adminclient-22] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:54 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------+-------+-----------+--------------------+
|hotel_id|   city|star_rating|       processing_ts|
+--------+-------+-----------+--------------------+
|      h1| Tehran|          4|2026-02-16 02:44:...|
|      h2| Shiraz|          5|2026-02-16 02:44:...|
|      h3|Isfahan|          3|2026-02-16 02:44:...|
|      h1| Tehran|          4|2026-02-16 05:27:...|
|      h2| Shiraz|          5|2026-02-16 05:27:...|
|      h3|Isfahan|          3|2026-02-16 05:27:...|
|      h1| Tehran|          4|2026-02-16 06:20:...|
|      h2| Shiraz|          5|2026-02-16 06:20:...|
|      h3|Isfahan|          3|2026-02-16 06:20:...|
|      h1| Tehran|          4|2026-02-21 14:05:...|
|      h2| Shiraz|          5|2026-02-21 14:05:...|
|      h3|Isfahan|          3|2026-02-21 14:05:...|
|      h1| Tehran|          4|2026-02-21 14:10:...|
|      h2| Shiraz|          5|2026-02-21 14:10:...|
|      h3|Isfahan|          3|2026-02-21 14:10:...|
|      h1| Tehran|          4|2026-02-21 14:12:...|
|      h2| S

26/02/21 21:21:55 WARN KafkaAdminClient: [AdminClient clientId=adminclient-23] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                


✅ Hotels written to HDFS Iceberg:
   📁 Path: hdfs://localhost:9000/lakehouse/reference/hotels
   📦 Format: Iceberg (Snappy Parquet)


26/02/21 21:21:57 WARN KafkaAdminClient: [AdminClient clientId=adminclient-24] Overriding the default value for default.api.timeout.ms (0) with the explicitly configured request timeout 120000
26/02/21 21:21:57 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


   📊 Records: 45


## 6. Verify Bronze Layer Tables in HDFS

In [15]:
import subprocess
_on_host = not os.environ.get("LAKEHOUSE_WAREHOUSE")
print("📋 Bronze Layer Tables:")
if _on_host:
    print("\n⚠️ Running on host: Spark catalog HDFS calls cause 'RPC response has invalid length'. Showing tables via HDFS CLI.\n")
    print("Bronze database (HDFS):")
    _r = subprocess.run(["docker", "exec", "namenode", "hdfs", "dfs", "-ls", "/lakehouse/bronze"], capture_output=True, text=True, timeout=10)
    print(_r.stdout if _r.returncode == 0 else _r.stderr or "(no output)")
    print("Reference database (HDFS):")
    _r2 = subprocess.run(["docker", "exec", "namenode", "hdfs", "dfs", "-ls", "/lakehouse/reference"], capture_output=True, text=True, timeout=10)
    print(_r2.stdout if _r2.returncode == 0 else _r2.stderr or "(no output)")
    print("\n💡 For full SHOW TABLES + schema/counts, run this notebook in Docker: docker compose -f docker/docker-compose.yml up -d notebook → http://localhost:8888")
else:
    print("\nBronze database:")
    spark.sql("SHOW TABLES IN local.bronze").show()
    print("\nReference database:")
    spark.sql("SHOW TABLES IN local.reference").show()
    print("\n📊 Bookings Table:")
    spark.table("local.bronze.bookings_raw").printSchema()
    print(f"Count: {spark.table('local.bronze.bookings_raw').count()}")
    print("\n📊 Events Table:")
    spark.table("local.bronze.booking_events_raw").printSchema()
    print(f"Count: {spark.table('local.bronze.booking_events_raw').count()}")
    print("\n📊 Hotels Table:")
    spark.table("local.reference.hotels").printSchema()
    print(f"Count: {spark.table('local.reference.hotels').count()}")

📋 Bronze Layer Tables:

Bronze database:
+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|   bronze|booking_events_raw|      false|
|   bronze|      bookings_raw|      false|
+---------+------------------+-----------+


Reference database:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|reference|   hotels|      false|
+---------+---------+-----------+


📊 Bookings Table:
root
 |-- booking_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- hotel_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- created_at_ts: timestamp (nullable = true)
 |-- updated_at_ts: timestamp (nullable = true)
 |-- processing_ts: timestamp (nullable = true)

Count: 150

📊 Events Table:
root
 |-- booking_id: string (n

## 7. Check HDFS Storage

In [16]:
import subprocess
import shutil

print("📁 HDFS Directory Structure:")
print("\n" + "=" * 70)

# List lakehouse (use Spark when in Docker—no 'docker' CLI; else use docker exec)
if not shutil.which("docker") and 'spark' in dir() and spark is not None:
    import os
    base = os.environ.get("LAKEHOUSE_WAREHOUSE", "hdfs://namenode:9000/lakehouse").rstrip("/")
    jvm = spark._jvm
    conf = jvm.org.apache.hadoop.conf.Configuration()
    conf.set("fs.defaultFS", "hdfs://namenode:9000")
    path = jvm.org.apache.hadoop.fs.Path(base)
    fs = path.getFileSystem(conf)
    def list_r(p, indent=""):
        p_path = p if hasattr(p, "getFileSystem") else jvm.org.apache.hadoop.fs.Path(p)
        for f in fs.listStatus(p_path):
            name = f.getPath().getName()
            print(indent + name)
            if f.isDirectory():
                list_r(f.getPath(), indent + "  ")
    try:
        list_r(path)
    except Exception as e:
        if "does not exist" in str(e) or "FileNotFoundException" in str(type(e).__name__):
            print(f"(Path {base} not found yet — run the ingestion cells above first.)")
        else:
            raise
else:
    result = subprocess.run(
        ['docker', 'exec', 'namenode', 'hdfs', 'dfs', '-ls', '-R', '/lakehouse'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr or "(no output)")

print("\n" + "=" * 70)
print("\n✅ Bronze layer data persisted in HDFS!")
print("\n📍 HDFS Paths:")
print("   • hdfs://namenode:9000/lakehouse/bronze/bookings_raw")
print("   • hdfs://namenode:9000/lakehouse/bronze/booking_events_raw")
print("   • hdfs://namenode:9000/lakehouse/reference/hotels")
print("\n🌐 View in HDFS UI: http://localhost:9870/explorer.html#/lakehouse")

📁 HDFS Directory Structure:

bronze
  booking_events_raw
    data
      event_ts_parsed=2025-01-01T10%3A05Z
        00000-109-8f0cda64-4a7d-48d2-9e4a-b6f048cf699a-00007.parquet
        00000-109-edb7a2fb-ea9d-4b80-9446-1b4e5db01bd8-00007.parquet
        00000-25-0f9ef4c4-89f1-4507-88c6-5e6bf5823292-00007.parquet
        00000-25-2ce1c8c2-b22c-4a54-80cb-7f80fb5c2762-00007.parquet
        00000-25-2cfa37dc-7951-4e08-9ceb-ca033a76b020-00007.parquet
        00000-25-38dc7a35-d197-4c0c-b4bc-522f2f27a5da-00007.parquet
        00000-25-53f95520-fdf7-46cb-b01f-e2bfb11fcd70-00007.parquet
        00000-25-6858afe6-30cf-49a5-931c-10f3443cfedc-00007.parquet
        00000-25-7ba3bc50-b3a3-4a8e-a562-93e39b7914fe-00007.parquet
        00000-25-a1900855-a751-4439-9f45-26ad3601e40a-00007.parquet
        00000-25-d1d5edd6-a3b3-4c9a-a7f1-05de9a9373f1-00007.parquet
        00000-25-e2acd165-593f-47cd-8e66-7d99662e0782-00007.parquet
        00000-25-ef645aa5-4ade-4233-a164-552ba5eeaed2-00007.parquet
      

## Summary

### Bronze Layer Created:
- ✅ **bookings_raw**: Ingested from Kafka, stored in HDFS Iceberg (Snappy Parquet)
- ✅ **booking_events_raw**: Ingested from Kafka, stored in HDFS Iceberg (Snappy Parquet)
- ✅ **hotels** (reference): Ingested from Kafka, stored in HDFS Iceberg (Snappy Parquet)

### Storage Details:
- **Location**: HDFS at `hdfs://namenode:9000/lakehouse`
- **Format**: Apache Iceberg
- **File Format**: Parquet
- **Compression**: Snappy

### Next Step:
Run **03_silver_layer.ipynb** to transform Bronze → Silver using DBT.